# Pipeline NLP — Phase 1 Compagnon IA

**Projet M1 Big Data — UGC et Mobile Marketing**

Pipeline complet d'analyse e-réputation sur deux corpus :
- **Corpus 1 — Apps santé** (Google Play + App Store) : ~32 000 reviews sur 11 apps concurrentes
- **Corpus 2 — Compagnons IA généralistes** (Reddit + Hacker News) : ~1 500 documents

## Structure du notebook

1. **Setup & chargement** — installation, imports, lecture CSV
2. **Module 1 — Nettoyage & prétraitement**
3. **Module 2 — Représentation** (TF-IDF + embeddings)
4. **Module 3 — Sentiment Analysis**
5. **Module 4 — Topic Modeling** (BERTopic)
6. **Module 5 — Classification thématique** (zero-shot)
7. **Module 6 — Visualisations & synthèse benchmark**

**Environnement recommandé : Google Colab avec GPU T4** (Runtime → Change runtime type → T4 GPU)

**Données attendues** : uploadez les 4 CSV dans Colab :
- `google_play_reviews.csv`
- `app_store_reviews.csv`
- `posts_reddit.csv`, `commentaires_reddit.csv`
- `hackernews.csv`

---
## 0. Setup — Installation des dépendances

*(Décommenter la première cellule au premier lancement, ~3 min)*

In [ ]:
# !pip install -q transformers sentence-transformers bertopic langdetect spacy seaborn wordcloud
# !python -m spacy download en_core_web_sm
# !python -m spacy download fr_core_news_sm

In [ ]:
# Imports
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from langdetect import detect, LangDetectException
from sklearn.feature_extraction.text import TfidfVectorizer

import torch
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

# Détection GPU
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"GPU disponible : {torch.cuda.is_available()}")
if DEVICE == 0:
    print(f"GPU : {torch.cuda.get_device_name(0)}")

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

---
## 1. Chargement & consolidation des données

On crée **deux corpus distincts** :
- `df_apps` : Google Play + App Store (normalisés en colonnes communes)
- `df_ia` : Reddit posts + comments + Hacker News

In [ ]:
# === CORPUS 1 : Apps santé (Google Play + App Store) ===

gp = pd.read_csv('google_play_reviews.csv')
ap = pd.read_csv('app_store_reviews.csv')

# Normalisation Google Play
gp_norm = pd.DataFrame({
    'source':    'google_play',
    'app':       gp['app'],
    'categorie': gp['categorie'],
    'langue':    gp['langue'],
    'note':      gp['note'],
    'texte':     gp['texte'].astype(str),
    'date':      gp['date'],
})

# Normalisation App Store
ap_norm = pd.DataFrame({
    'source':    'app_store',
    'app':       ap['app'],
    'categorie': ap['categorie'],
    'langue':    ap['pays'].map({'fr': 'fr'}).fillna('en'),  # approximation
    'note':      ap['note'],
    'texte':     ap['texte'].astype(str),
    'date':      None,
})

df_apps = pd.concat([gp_norm, ap_norm], ignore_index=True)
df_apps = df_apps[df_apps['texte'].str.len() > 5].reset_index(drop=True)

print(f"📱 CORPUS APPS : {len(df_apps):,} reviews")
print(f"\nPar source :\n{df_apps['source'].value_counts()}")
print(f"\nPar app :\n{df_apps.groupby(['categorie', 'app']).size()}")

In [ ]:
# === CORPUS 2 : Compagnons IA généralistes (Reddit + HN) ===

posts = pd.read_csv('posts_reddit.csv')
comm  = pd.read_csv('commentaires_reddit.csv')
hn    = pd.read_csv('hackernews.csv')

# Normalisation posts Reddit (titre + texte fusionnés)
posts_norm = pd.DataFrame({
    'source':     'reddit_post',
    'plateforme': posts['subreddit'],
    'categorie':  posts['categorie'],
    'texte':      (posts['titre'].fillna('').astype(str) + ' ' + posts['texte'].fillna('').astype(str)).str.strip(),
    'score':      posts['score'],
})

# Normalisation commentaires Reddit
comm_norm = pd.DataFrame({
    'source':     'reddit_comment',
    'plateforme': comm['subreddit'],
    'categorie':  comm['categorie'],
    'texte':      comm['texte'].astype(str),
    'score':      comm['score'],
})

# Normalisation HN
hn_norm = pd.DataFrame({
    'source':     'hackernews',
    'plateforme': hn['requete'],
    'categorie':  'general_ia',
    'texte':      hn['texte'].astype(str),
    'score':      hn['points'],
})

df_ia = pd.concat([posts_norm, comm_norm, hn_norm], ignore_index=True)
df_ia = df_ia[df_ia['texte'].str.len() > 30].reset_index(drop=True)

print(f"🤖 CORPUS COMPAGNONS IA : {len(df_ia):,} documents")
print(f"\nPar source :\n{df_ia['source'].value_counts()}")

---
## 2. Module 1 — Nettoyage & prétraitement

Étapes appliquées aux deux corpus :
- Suppression doublons exacts
- Détection langue (filtrage en/fr uniquement)
- Nettoyage texte : URLs, mentions, caractères spéciaux
- Création d'une colonne `texte_clean` (pour modèles transformers, qui aiment le texte naturel)
- Création d'une colonne `texte_lemma` (pour TF-IDF et word clouds)

In [ ]:
import spacy

# Charger spaCy (en + fr)
nlp_en = spacy.load('en_core_web_sm', disable=['ner', 'parser'])
nlp_fr = spacy.load('fr_core_news_sm', disable=['ner', 'parser'])

URL_RE = re.compile(r'http\S+|www\.\S+')
MENTION_RE = re.compile(r'[@#]\w+')
MULTISPACE_RE = re.compile(r'\s+')
EMOJI_RE = re.compile("[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+", flags=re.UNICODE)

def clean_text(t):
    """Nettoyage léger pour modèles transformers (préserve la ponctuation et la sémantique)."""
    if not isinstance(t, str):
        return ''
    t = URL_RE.sub('', t)
    t = MENTION_RE.sub('', t)
    t = EMOJI_RE.sub('', t)
    t = MULTISPACE_RE.sub(' ', t).strip()
    return t

def detect_lang_safe(t):
    if not t or len(t) < 20:
        return 'unknown'
    try:
        return detect(t)
    except LangDetectException:
        return 'unknown'

def lemmatize(t, lang):
    """Lemmatisation lourde pour TF-IDF (lower + lemma + stopwords retirés)."""
    if not t or len(t) < 5:
        return ''
    nlp = nlp_fr if lang == 'fr' else nlp_en
    doc = nlp(t.lower()[:1000])  # tronquer pour vitesse
    return ' '.join(tok.lemma_ for tok in doc 
                    if not tok.is_stop and not tok.is_punct 
                    and not tok.is_space and len(tok.lemma_) > 2)

In [ ]:
# === Nettoyage corpus APPS ===
print('Nettoyage corpus APPS...')

df_apps['texte_clean'] = df_apps['texte'].apply(clean_text)
df_apps = df_apps[df_apps['texte_clean'].str.len() > 10].drop_duplicates(subset=['texte_clean']).reset_index(drop=True)

# Si la colonne langue existe déjà (Google Play), on l'utilise. Sinon on détecte.
mask_unknown = df_apps['langue'].isna() | (df_apps['langue'] == 'unknown')
df_apps.loc[mask_unknown, 'langue'] = df_apps.loc[mask_unknown, 'texte_clean'].apply(detect_lang_safe)

# Garder uniquement en + fr
df_apps = df_apps[df_apps['langue'].isin(['en', 'fr'])].reset_index(drop=True)

print(f"✓ Après nettoyage : {len(df_apps):,} reviews")
print(f"  EN : {(df_apps['langue']=='en').sum():,}  |  FR : {(df_apps['langue']=='fr').sum():,}")

# Lemmatisation (long, ~5 min pour 30k reviews)
print('\nLemmatisation en cours (~5 min)...')
df_apps['texte_lemma'] = [lemmatize(t, l) for t, l in zip(df_apps['texte_clean'], df_apps['langue'])]
print('✓ Lemmatisation terminée')
df_apps.head(2)

In [ ]:
# === Nettoyage corpus IA ===
print('Nettoyage corpus IA...')

df_ia['texte_clean'] = df_ia['texte'].apply(clean_text)
df_ia = df_ia[df_ia['texte_clean'].str.len() > 30].drop_duplicates(subset=['texte_clean']).reset_index(drop=True)

df_ia['langue'] = df_ia['texte_clean'].apply(detect_lang_safe)
df_ia = df_ia[df_ia['langue'].isin(['en', 'fr'])].reset_index(drop=True)

print(f"✓ Après nettoyage : {len(df_ia):,} documents")
print(f"  EN : {(df_ia['langue']=='en').sum():,}  |  FR : {(df_ia['langue']=='fr').sum():,}")

print('\nLemmatisation en cours...')
df_ia['texte_lemma'] = [lemmatize(t, l) for t, l in zip(df_ia['texte_clean'], df_ia['langue'])]
print('✓ Lemmatisation terminée')

---
## 3. Module 2 — Représentation

**Deux représentations en parallèle**, chacune utile pour des tâches différentes :

| Représentation | Outil | Usage |
|---|---|---|
| **TF-IDF** | scikit-learn | Word clouds, top mots par app, mots discriminants |
| **Embeddings** | sentence-transformers (multilingue) | Topic modeling (BERTopic), similarité sémantique |

In [ ]:
# === TF-IDF ===
print('Calcul TF-IDF...')

tfidf_apps = TfidfVectorizer(max_features=2000, ngram_range=(1,2), min_df=5)
X_tfidf_apps = tfidf_apps.fit_transform(df_apps['texte_lemma'].fillna(''))
print(f"✓ TF-IDF apps : {X_tfidf_apps.shape}")

tfidf_ia = TfidfVectorizer(max_features=1500, ngram_range=(1,2), min_df=3)
X_tfidf_ia = tfidf_ia.fit_transform(df_ia['texte_lemma'].fillna(''))
print(f"✓ TF-IDF IA : {X_tfidf_ia.shape}")

In [ ]:
# === Embeddings (multilingue, ~10 min sur GPU) ===
print('Chargement modèle embeddings multilingue...')
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device='cuda' if DEVICE == 0 else 'cpu')

print('\nEmbeddings corpus APPS (peut prendre 5-10 min)...')
emb_apps = embedder.encode(df_apps['texte_clean'].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True)
print(f"✓ Embeddings apps : {emb_apps.shape}")

print('\nEmbeddings corpus IA...')
emb_ia = embedder.encode(df_ia['texte_clean'].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True)
print(f"✓ Embeddings IA : {emb_ia.shape}")

---
## 4. Module 3 — Sentiment Analysis

Modèle : **`cardiffnlp/twitter-xlm-roberta-base-sentiment`** (multilingue en/fr, robuste sur du contenu user-generated)

Sortie : 3 classes **negative / neutral / positive** avec score de confiance.

Sur le corpus apps, on **valide** notre sentiment en le croisant avec la note ★ (vérité terrain partielle) → si forte corrélation, le modèle est fiable.

In [ ]:
print('Chargement modèle de sentiment...')
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-xlm-roberta-base-sentiment',
    device=DEVICE,
    truncation=True,
    max_length=512,
)

def analyser_sentiment(textes, batch_size=64):
    """Sentiment par batch."""
    results = []
    for i in range(0, len(textes), batch_size):
        batch = [t[:512] for t in textes[i:i+batch_size]]
        try:
            res = sentiment_pipe(batch)
            results.extend(res)
        except Exception as e:
            print(f'Erreur batch {i}: {e}')
            results.extend([{'label': 'neutral', 'score': 0.0}] * len(batch))
        if i % (batch_size * 10) == 0:
            print(f'  {i}/{len(textes)} traités')
    return results

In [ ]:
# === Sentiment corpus APPS ===
print('Sentiment corpus APPS (~10-15 min sur GPU)...')
sent_apps = analyser_sentiment(df_apps['texte_clean'].tolist())
df_apps['sentiment'] = [r['label'] for r in sent_apps]
df_apps['sentiment_score'] = [r['score'] for r in sent_apps]

# Score numérique pour agrégation : -1 / 0 / +1 pondéré par confiance
label_to_num = {'negative': -1, 'neutral': 0, 'positive': 1}
df_apps['sentiment_num'] = df_apps['sentiment'].map(label_to_num) * df_apps['sentiment_score']

print(f"\nDistribution sentiment apps :\n{df_apps['sentiment'].value_counts(normalize=True).round(3)}")

In [ ]:
# Validation : sentiment vs note ★ (corrélation attendue)
print('VALIDATION DU MODÈLE — sentiment vs note utilisateur :\n')
validation = df_apps.groupby('note')['sentiment_num'].mean().round(3)
print(validation)
print(f"\nCorrélation Pearson (note ↔ sentiment_num) : {df_apps[['note', 'sentiment_num']].corr().iloc[0,1]:.3f}")
print("→ Si > 0.5, le modèle de sentiment est cohérent avec les notes utilisateurs ✓")

In [ ]:
# === Sentiment corpus IA ===
print('Sentiment corpus IA...')
sent_ia = analyser_sentiment(df_ia['texte_clean'].tolist())
df_ia['sentiment'] = [r['label'] for r in sent_ia]
df_ia['sentiment_score'] = [r['score'] for r in sent_ia]
df_ia['sentiment_num'] = df_ia['sentiment'].map(label_to_num) * df_ia['sentiment_score']

print(f"\nDistribution sentiment IA :\n{df_ia['sentiment'].value_counts(normalize=True).round(3)}")

---
## 5. Module 4 — Topic Modeling (BERTopic)

**Découverte automatique de thèmes** dans chaque corpus, basée sur les embeddings.

On va découvrir :
- Sur les apps : *« paywall agressif »*, *« bug de connexion »*, *« réponses répétitives »*, *« sentiment d'écoute »*, etc.
- Sur les compagnons IA : *« qualité du raisonnement »*, *« hallucinations »*, *« attachement émotionnel »*, etc.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Vectorizer custom pour BERTopic (top mots par topic)
vec_model = CountVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))

# === Topics APPS ===
print('BERTopic corpus APPS (~5 min)...')
topic_model_apps = BERTopic(
    embedding_model=embedder,
    vectorizer_model=vec_model,
    min_topic_size=30,
    nr_topics='auto',
    verbose=False,
)
topics_apps, probs_apps = topic_model_apps.fit_transform(df_apps['texte_clean'].tolist(), embeddings=emb_apps)
df_apps['topic'] = topics_apps

print(f"\n✓ {len(topic_model_apps.get_topic_info()) - 1} thèmes découverts (hors outliers)")
print('\nTop 10 thèmes :')
topic_model_apps.get_topic_info().head(11)

In [ ]:
# Affichage détaillé des top topics avec mots-clés
for tid in topic_model_apps.get_topic_info().head(11)['Topic'].values:
    if tid == -1:
        continue
    mots = [w for w, _ in topic_model_apps.get_topic(tid)[:8]]
    n = (df_apps['topic'] == tid).sum()
    print(f"Topic {tid:2d} ({n:>5} docs) : {', '.join(mots)}")

In [ ]:
# === Topics IA ===
print('BERTopic corpus IA...')
vec_model_ia = CountVectorizer(stop_words='english', min_df=3, ngram_range=(1, 2))

topic_model_ia = BERTopic(
    embedding_model=embedder,
    vectorizer_model=vec_model_ia,
    min_topic_size=10,
    nr_topics='auto',
    verbose=False,
)
topics_ia, probs_ia = topic_model_ia.fit_transform(df_ia['texte_clean'].tolist(), embeddings=emb_ia)
df_ia['topic'] = topics_ia

print(f"\n✓ {len(topic_model_ia.get_topic_info()) - 1} thèmes découverts")
for tid in topic_model_ia.get_topic_info().head(11)['Topic'].values:
    if tid == -1:
        continue
    mots = [w for w, _ in topic_model_ia.get_topic(tid)[:8]]
    n = (df_ia['topic'] == tid).sum()
    print(f"Topic {tid:2d} ({n:>4} docs) : {', '.join(mots)}")

---
## 6. Module 5 — Classification thématique (zero-shot)

Au lieu de laisser BERTopic découvrir, on **classe explicitement** chaque review dans des catégories utiles pour le benchmark :

- `bug_technique` — plantages, lenteurs, problèmes de connexion
- `prix_paywall` — coût, abonnement, fonctionnalités payantes
- `qualite_conversation` — pertinence des réponses, ton, écoute
- `empathie` — chaleur, soutien émotionnel, sensation d'être compris
- `fonctionnalite_manquante` — ce qui n'est pas dispo
- `interface_ux` — design, ergonomie, navigation
- `utilite_perçue` — bénéfice ressenti, impact réel

Modèle : **`facebook/bart-large-mnli`** (zero-shot, pas besoin d'entraîner).

In [ ]:
print('Chargement modèle zero-shot...')
zsc = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=DEVICE,
)

CATEGORIES = [
    'technical bug or crash',
    'price or paywall complaint',
    'conversation quality',
    'empathy and emotional support',
    'missing feature',
    'interface and user experience',
    'perceived usefulness',
]

CAT_LABELS = {
    'technical bug or crash':         'bug_technique',
    'price or paywall complaint':     'prix_paywall',
    'conversation quality':           'qualite_conversation',
    'empathy and emotional support':  'empathie',
    'missing feature':                'fonctionnalite_manquante',
    'interface and user experience':  'interface_ux',
    'perceived usefulness':           'utilite_percue',
}

def classify_zsc(textes, batch_size=16):
    """Classifie chaque texte dans la catégorie la plus probable."""
    cats = []
    for i in range(0, len(textes), batch_size):
        batch = [t[:512] for t in textes[i:i+batch_size]]
        try:
            results = zsc(batch, candidate_labels=CATEGORIES, multi_label=False)
            if isinstance(results, dict):
                results = [results]
            for r in results:
                cats.append(CAT_LABELS[r['labels'][0]])
        except Exception as e:
            print(f'Erreur batch {i}: {e}')
            cats.extend(['unknown'] * len(batch))
        if i % (batch_size * 5) == 0:
            print(f'  {i}/{len(textes)} classifiés')
    return cats

In [ ]:
# Classification corpus APPS (zero-shot est lent, on échantillonne pour gagner du temps)
# Sur GPU : ~25 min pour 30k. On peut échantillonner si trop long.
ECHANTILLON_APPS = min(15000, len(df_apps))   # mets None pour tout traiter

if ECHANTILLON_APPS < len(df_apps):
    print(f'Échantillonnage : {ECHANTILLON_APPS} reviews stratifiées par app')
    df_apps_sample = df_apps.groupby('app', group_keys=False).apply(
        lambda x: x.sample(min(len(x), ECHANTILLON_APPS // df_apps['app'].nunique()), random_state=42)
    ).reset_index(drop=True)
else:
    df_apps_sample = df_apps.copy()

print(f'\nClassification {len(df_apps_sample)} reviews (~10-20 min)...')
df_apps_sample['theme'] = classify_zsc(df_apps_sample['texte_clean'].tolist())

print(f"\nDistribution thèmes apps :\n{df_apps_sample['theme'].value_counts()}")

In [ ]:
# Classification corpus IA (volume faible, on peut tout classer)
print(f'Classification corpus IA ({len(df_ia)} docs)...')
df_ia['theme'] = classify_zsc(df_ia['texte_clean'].tolist())
print(f"\nDistribution thèmes IA :\n{df_ia['theme'].value_counts()}")

---
## 7. Module 6 — Visualisations & synthèse benchmark

Tous les graphiques pour le rapport et les slides.

In [ ]:
# === GRAPHIQUE 1 : Score sentiment moyen par app ===
fig, ax = plt.subplots(figsize=(11, 6))
app_sent = df_apps.groupby(['categorie', 'app'])['sentiment_num'].mean().sort_values()
colors = ['#e74c3c' if v < 0 else ('#f39c12' if v < 0.2 else '#2ecc71') for v in app_sent.values]
app_sent.plot.barh(color=colors, ax=ax)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Score sentiment moyen (-1 = négatif, +1 = positif)')
ax.set_title('E-réputation comparative des apps santé concurrentes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_sentiment_par_app.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === GRAPHIQUE 2 : Distribution sentiment par app (empilé) ===
pivot = df_apps.groupby(['app', 'sentiment']).size().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct = pivot_pct[['negative', 'neutral', 'positive']]

fig, ax = plt.subplots(figsize=(11, 6))
pivot_pct.sort_values('positive').plot.barh(
    stacked=True, ax=ax,
    color=['#e74c3c', '#bdc3c7', '#2ecc71']
)
ax.set_xlabel('% des reviews')
ax.set_title('Répartition sentiment par app', fontsize=13, fontweight='bold')
ax.legend(title='Sentiment', loc='lower right')
plt.tight_layout()
plt.savefig('viz_sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === GRAPHIQUE 3 : Heatmap App × Thème ===
heatmap_data = df_apps_sample.groupby(['app', 'theme']).size().unstack(fill_value=0)
heatmap_pct = heatmap_data.div(heatmap_data.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(heatmap_pct, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, cbar_kws={'label': '% des reviews'})
ax.set_title('Thématiques discutées par app (% des reviews)', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('viz_heatmap_themes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === GRAPHIQUE 4 : Sentiment moyen par thème (les sujets les plus controversés) ===
theme_sent = df_apps_sample.groupby('theme')['sentiment_num'].agg(['mean', 'count']).sort_values('mean')

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if v < 0 else ('#f39c12' if v < 0.2 else '#2ecc71') for v in theme_sent['mean']]
ax.barh(theme_sent.index, theme_sent['mean'], color=colors)
ax.axvline(0, color='black', linewidth=0.5)
for i, (mean, count) in enumerate(zip(theme_sent['mean'], theme_sent['count'])):
    ax.text(mean, i, f'  n={count}', va='center', fontsize=9)
ax.set_xlabel('Sentiment moyen')
ax.set_title('Quels thèmes génèrent le plus de frustration ?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_sentiment_par_theme.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === GRAPHIQUE 5 : Word clouds par catégorie ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, cat in zip(axes, ['sante_mentale', 'maladie_chronique']):
    txt = ' '.join(df_apps[df_apps['categorie'] == cat]['texte_lemma'].fillna('').tolist())
    wc = WordCloud(width=800, height=400, background_color='white', 
                   max_words=100, colormap='viridis').generate(txt)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'Mots-clés — {cat}', fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('viz_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === TABLEAU DE SYNTHÈSE BENCHMARK ===
synthese = df_apps.groupby(['categorie', 'app']).agg(
    n_reviews=('texte_clean', 'count'),
    note_moyenne=('note', 'mean'),
    sentiment_moyen=('sentiment_num', 'mean'),
    pct_negatif=('sentiment', lambda x: (x == 'negative').mean() * 100),
    pct_positif=('sentiment', lambda x: (x == 'positive').mean() * 100),
).round(2)

# Top thème + thème négatif dominant par app
top_themes = df_apps_sample.groupby('app')['theme'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'N/A'
)

neg_themes = df_apps_sample[df_apps_sample['sentiment'] == 'negative'].groupby('app')['theme'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'N/A'
)

synthese['theme_dominant'] = synthese.index.get_level_values('app').map(top_themes)
synthese['frustration_principale'] = synthese.index.get_level_values('app').map(neg_themes)

print('=== TABLEAU DE SYNTHÈSE BENCHMARK ===\n')
print(synthese.to_string())

# Export
synthese.to_csv('benchmark_synthese.csv', encoding='utf-8-sig')
print('\n✓ Exporté : benchmark_synthese.csv')

In [ ]:
# === ANALYSE CORPUS COMPAGNONS IA ===
print('=== ANALYSE COMPAGNONS IA GÉNÉRALISTES ===\n')

# Sentiment par plateforme/source
ia_sent = df_ia.groupby('source')['sentiment_num'].agg(['mean', 'count']).round(3)
print('Sentiment par source :')
print(ia_sent)

# Thèmes dominants
print('\nThèmes dominants (corpus IA) :')
print(df_ia['theme'].value_counts())

# Sentiment par thème
print('\nSentiment par thème (corpus IA) :')
print(df_ia.groupby('theme')['sentiment_num'].agg(['mean', 'count']).sort_values('mean').round(3))

In [ ]:
# === GRAPHIQUE FINAL : comparaison thèmes apps vs compagnons IA ===
df_apps_sample['corpus'] = 'Apps santé'
df_ia['corpus'] = 'Compagnons IA'

compar = pd.concat([df_apps_sample[['corpus', 'theme', 'sentiment_num']],
                     df_ia[['corpus', 'theme', 'sentiment_num']]])

pct = compar.groupby(['corpus', 'theme']).size().reset_index(name='n')
pct['pct'] = pct.groupby('corpus')['n'].transform(lambda x: x / x.sum() * 100)

fig, ax = plt.subplots(figsize=(11, 5))
pct.pivot(index='theme', columns='corpus', values='pct').plot.bar(ax=ax)
ax.set_ylabel('% des reviews/posts')
ax.set_title('Préoccupations exprimées : Apps santé vs Compagnons IA généralistes', fontsize=12, fontweight='bold')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('viz_comparaison_corpus.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === SAUVEGARDE FINALE ===
df_apps.to_csv('corpus_apps_enrichi.csv', index=False, encoding='utf-8-sig')
df_apps_sample.to_csv('corpus_apps_avec_themes.csv', index=False, encoding='utf-8-sig')
df_ia.to_csv('corpus_ia_enrichi.csv', index=False, encoding='utf-8-sig')

print('=' * 60)
print('PIPELINE NLP TERMINÉ')
print('=' * 60)
print('\nFichiers générés :')
print('  • corpus_apps_enrichi.csv          (toutes les apps avec sentiment)')
print('  • corpus_apps_avec_themes.csv      (échantillon avec thèmes zero-shot)')
print('  • corpus_ia_enrichi.csv            (compagnons IA avec sentiment + thèmes)')
print('  • benchmark_synthese.csv           (tableau de synthèse pour le rapport)')
print('\nVisualisations PNG :')
print('  • viz_sentiment_par_app.png')
print('  • viz_sentiment_distribution.png')
print('  • viz_heatmap_themes.png')
print('  • viz_sentiment_par_theme.png')
print('  • viz_wordclouds.png')
print('  • viz_comparaison_corpus.png')